# Markerless motion capture — you don't need the rig

Yesterday you used a **marker-based** system (OptiTrack): bright markers, a ring of cameras, a lab. Here you'll do motion capture with **an ordinary video and free software** — Google's **MediaPipe** finds your body joints in every frame, *no markers*.

**How to use this:** run the cells top-to-bottom (they work on a provided sample). Then set the video source to **upload** and try it on **your own** movement — a squat, a reach, walking past the camera.

*Companion to the Day-1/Day-2 table-wiping notebooks. For "is markerless accurate enough?", see the `markerless_vs_rig` comparison on the Day-1 data.*

## Setup

In [ ]:
#@title Setup — install + load MediaPipe { display-mode: "form" }
# On Colab the installs run once (~30 s). Re-run this cell if you restart the runtime.
try:
    import google.colab; IN_COLAB = True
except Exception:
    IN_COLAB = False
import subprocess, sys, os, tempfile, urllib.request
try:
    import mediapipe, cv2
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe", "opencv-python-headless"], check=False)
import numpy as np, cv2, mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.core.base_options import BaseOptions
import matplotlib.pyplot as plt

MODEL = os.path.join(tempfile.gettempdir(), "pose_landmarker_full.task")
if not os.path.exists(MODEL):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
        "pose_landmarker_full/float16/latest/pose_landmarker_full.task", MODEL)

# BlazePose 33 landmarks: the skeleton edges (for drawing) and 3-point joints (for angles)
CONN = [(11,12),(11,23),(12,24),(23,24),(11,13),(13,15),(12,14),(14,16),
        (23,25),(25,27),(27,29),(24,26),(26,28),(28,30),(15,17),(16,18),(0,11),(0,12)]
JOINTS = {"right elbow":(12,14,16),"left elbow":(11,13,15),"right knee":(24,26,28),
          "left knee":(23,25,27),"right shoulder":(14,12,24),"left shoulder":(13,11,23)}

def run_pose(video):
    """Return per-frame landmarks (N,33,3 = x_px,y_px,visibility), the frames, fps, (W,H)."""
    cap = cv2.VideoCapture(video); fps = cap.get(cv2.CAP_PROP_FPS)
    W, H = int(cap.get(3)), int(cap.get(4))
    opts = vision.PoseLandmarkerOptions(base_options=BaseOptions(model_asset_path=MODEL),
            running_mode=vision.RunningMode.VIDEO, num_poses=1)
    LM, frames, i = [], [], 0
    with vision.PoseLandmarker.create_from_options(opts) as lm:
        while True:
            ok, fr = cap.read()
            if not ok: break
            res = lm.detect_for_video(
                mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)),
                int(i / fps * 1000))
            LM.append(np.array([[p.x*W, p.y*H, p.visibility] for p in res.pose_landmarks[0]])
                      if res.pose_landmarks else np.full((33,3), np.nan))
            frames.append(fr); i += 1
    cap.release()
    return np.array(LM), frames, fps, (W, H)

def draw(frame, pts):
    a = frame.copy()
    for u, v in CONN:
        if not (np.isnan(pts[u,0]) or np.isnan(pts[v,0])):
            cv2.line(a, tuple(pts[u,:2].astype(int)), tuple(pts[v,:2].astype(int)), (0,230,120), 2)
    for j in pts:
        if not np.isnan(j[0]): cv2.circle(a, tuple(j[:2].astype(int)), 3, (255,255,255), -1)
    return a

def joint_angle(LM, joint):
    a, b, c = JOINTS[joint]
    ba = LM[:,a,:2]-LM[:,b,:2]; bc = LM[:,c,:2]-LM[:,b,:2]
    cos = (ba*bc).sum(-1)/(np.linalg.norm(ba,axis=-1)*np.linalg.norm(bc,axis=-1)+1e-9)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

def count_reps(sig, fps, min_sep_s=0.4):
    s = np.nan_to_num(sig - np.nanmean(sig))
    k = max(1, int(fps*0.1)); s = np.convolve(s, np.ones(k)/k, "same")
    thr = 0.3*np.nanstd(s); peaks, last = [], -1e9
    for i in range(1, len(s)-1):
        if s[i] > s[i-1] and s[i] >= s[i+1] and s[i] > thr and (i-last)/fps >= min_sep_s:
            peaks.append(i); last = i
    return peaks

print("MediaPipe", mp.__version__, "ready.   IN_COLAB =", IN_COLAB)

## 1 · Get a video
Run as-is to use the sample (a clip from the Day-1 capture), or set `source = "upload"` on Colab to use your own.

In [ ]:
#@title Get a video — upload your own, or use the sample { display-mode: "form" }
source = "sample"  #@param ["sample", "upload"]
SAMPLE_URL = "https://raw.githubusercontent.com/praneethnamburi/immersionlab-pe-mis/main/notebooks/assets/markerless_sample.mp4"
if source == "upload" and IN_COLAB:
    from google.colab import files
    VIDEO = list(files.upload().keys())[0]
else:
    VIDEO = "assets/markerless_sample.mp4" if os.path.exists("assets/markerless_sample.mp4") \
            else os.path.join(tempfile.gettempdir(), "markerless_sample.mp4")
    if not os.path.exists(VIDEO):
        urllib.request.urlretrieve(SAMPLE_URL, VIDEO)
print("video:", VIDEO)

## 2 · Run markerless pose
MediaPipe finds 33 body landmarks in every frame — no markers, one camera. Below: the skeleton on your video.

In [ ]:
#@title Run markerless pose — the skeleton on your video { display-mode: "form" }
LM, frames, fps, (W, H) = run_pose(VIDEO)
tracked = float(np.mean(~np.isnan(LM[:,0,0]))*100)
print(f"{len(frames)} frames @ {fps:.0f} fps  ·  pose found in {tracked:.0f}% of frames")

out = os.path.join(tempfile.gettempdir(), "annotated.mp4")
vw = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
for fr, pts in zip(frames, LM): vw.write(draw(fr, pts))
vw.release()

shown = False   # try to embed a playable clip; fall back to a grid of frames
try:
    h264 = os.path.join(tempfile.gettempdir(), "annotated_h264.mp4")
    subprocess.run(["ffmpeg","-y","-i",out,"-vcodec","libx264","-pix_fmt","yuv420p",h264],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    from IPython.display import Video, display
    display(Video(h264, embed=True, width=480)); shown = True
except Exception:
    pass
if not shown:
    idx = np.linspace(0, len(frames)-1, 6).astype(int)
    fig, ax = plt.subplots(2, 3, figsize=(12, 5))
    for k, ii in enumerate(idx):
        ax.flat[k].imshow(cv2.cvtColor(draw(frames[ii], LM[ii]), cv2.COLOR_BGR2RGB)); ax.flat[k].axis("off")
    plt.tight_layout(); plt.show()

## 3 · From skeleton to a number
A skeleton is only useful once it becomes a measurement. Pick a joint; we compute its angle over time and count reps/cycles automatically.

In [ ]:
#@title From skeleton to a number — a joint angle + rep count { display-mode: "form" }
joint = "right elbow"  #@param ["right elbow","left elbow","right knee","left knee","right shoulder","left shoulder"]
ang = joint_angle(LM, joint)
t = np.arange(len(ang))/fps
reps = count_reps(ang, fps)
plt.figure(figsize=(9, 3.2))
plt.plot(t, ang, color="#0b7a52", lw=2)
plt.plot(t[reps], ang[reps], "o", color="#c026d3", label=f"{len(reps)} reps / cycles")
plt.xlabel("time (s)"); plt.ylabel(f"{joint} angle (deg)")
plt.title(f"{joint}: {np.nanmin(ang):.0f}-{np.nanmax(ang):.0f} deg  ·  {len(reps)} reps/cycles")
plt.legend(); plt.grid(alpha=.2); plt.tight_layout(); plt.show()
print(f"{joint}: range {np.nanmin(ang):.0f}-{np.nanmax(ang):.0f} deg;  {len(reps)} reps/cycles")

## Is it good enough? — the honest part

Single-camera markerless is **excellent** for **gross movement, timing, coordination, and counting** under good conditions — and it's **free, phone-portable, deployable anywhere**. It's **weaker** for absolute 3-D precision, contact/force, and fast or occluded motion.

The professional's question isn't *"is it as good as the rig?"* — it's **"is it good enough for _my_ application?"** Ask:
1. **What must I measure, how precisely?** angles / timing / counts → yes; mm-level 3-D → often no.
2. **My conditions?** good light, unobstructed → thrives; occlusion / speed → degrades.
3. **My accuracy budget?** would a 5° error change my decision? no → fine.
4. **Scale / cost / deployment?** field, many sites, no lab → markerless wins.
5. **Validate once, then trust:** check markerless vs a reference on a sample of your task, characterize the error, deploy within that budget.

On the Day-1 table-wiping, single-phone MediaPipe recovered the **same 30 wiping strokes** as the OptiTrack rig (r ≈ 0.8) — same movement structure and timing, noisier only in the messy start. *Good enough to count strokes and read timing; not for sub-millimeter kinematics.*

## Design your own / your turn

- **Track a different joint** — change `joint` above: knee (squats), shoulder (reaches), elbow (curls).
- **Count your reps** — the same peak-counter works on any cyclic movement; check it against your own count.
- **Compute speed** — `LM[:, 16, :2]` is the right-wrist path; the frame-to-frame distance is its speed. Where is it fastest?
- **Compare two clips** — record a "clean" and a "sloppy" rep; overlay the two angle curves.
- **On the day** — run this on **one of the two OpenCap phone videos**, then compare the single-phone MediaPipe read to the two-phone OpenCap 3-D of the *same* movement.

*Vibe-code freely — ask the assistant to modify any cell.*